# K-Means Clustering on Stock Prices Dataset
In this notebook, we'll apply K-Means clustering to a stock prices dataset to group stocks based on their annualized return and volatility. This is a common approach in finance for identifying stocks with similar risk-return profiles.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import os

%matplotlib inline
sns.set(style='whitegrid')

# Create plots directory if it doesn't exist
os.makedirs('plots', exist_ok=True)

## 1. Load Dataset
We load the dataset from the `datasets` folder and prepare the features.

In [ ]:
dataset_path = 'G:\Projects\CodVeda\machine-learning-models-portfolio\stock_market_clustering\stock_prices_dataset.csv'
df = pd.read_csv(dataset_path)
df['date'] = pd.to_datetime(df['date'])
df.head()

## 2. Preprocess Data
For clustering, we calculate the daily return for each stock, and then annualize the mean return and volatility.

In [ ]:
# Sort values by symbol and date
df.sort_values(by=['symbol', 'date'], inplace=True)

# Calculate daily return for each stock
df['daily_return'] = df.groupby('symbol')['close'].pct_change()

# Calculate mean daily return and volatility
stock_stats = df.groupby('symbol').agg(
    mean_return=('daily_return', 'mean'),
    volatility=('daily_return', 'std')
).dropna()

# Annualize returns and volatility (assuming 252 trading days/year)
stock_stats['annual_return'] = stock_stats['mean_return'] * 252
stock_stats['annual_volatility'] = stock_stats['volatility'] * np.sqrt(252)

# Features array for clustering
X = stock_stats[['annual_return', 'annual_volatility']]

# Display some stats
stock_stats[['annual_return', 'annual_volatility']].head()

We need to scale the data because the return and volatility features might be on slightly different scales.

In [ ]:
# Scalling data using StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## 3. Apply K-Means and Determine Optimal K
We use the **Elbow Method** to calculate the Within-Cluster Sum of Square (WCSS) for multiple values of K, helping us select the optimal count.

In [ ]:
wcss = []
k_values = range(1, 11)

for i in k_values:
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

# Plotting the elbow curve
plt.figure(figsize=(10, 6))
plt.plot(k_values, wcss, marker='o', linestyle='--')
plt.title('Elbow Method For Optimal k')
plt.xlabel('Number of clusters (k)')
plt.ylabel('WCSS (Within-Cluster Sum of Square)')
plt.grid(True)
plt.savefig('plots/elbow_method.png')
plt.show()

From the elbow plot, usually k=4 provides a reasonable trade-off. We will map the clusters onto the dataframe.

In [ ]:
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
stock_stats['cluster'] = kmeans.fit_predict(X_scaled)

## 4. Visualize Clusters
We use a 2D scatter plot to visualize how the stocks are grouped across default expected returns and volatility profiles.

In [ ]:
plt.figure(figsize=(12, 8))
sns.scatterplot(
    data=stock_stats, 
    x='annual_volatility', 
    y='annual_return', 
    hue='cluster', 
    palette='viridis', 
    s=100, 
    alpha=0.7
)

# Plot centroids
centroids = scaler.inverse_transform(kmeans.cluster_centers_)
plt.scatter(
    centroids[:, 1], 
    centroids[:, 0], 
    s=300, 
    c='red', 
    marker='X', 
    label='Centroids'
)

plt.title(f'Stock Market Clustering (K-Means, k={optimal_k})')
plt.xlabel('Annualized Volatility (Risk)')
plt.ylabel('Annualized Return')
plt.legend()
plt.grid(True)
plt.savefig('plots/stock_clusters.png')
plt.show()

## 5. Interpret Clustering Results
Let's look at the average return and volatility of each cluster to classify them (e.g., High-Risk High-Reward, Low-Risk Low-Reward, etc.).

In [ ]:
for i in range(optimal_k):
    cluster_data = stock_stats[stock_stats['cluster'] == i]
    print(f"\n--- Cluster {i} ---")
    print(f"Count: {len(cluster_data)} stocks")
    print(f"Average Annual Return: {cluster_data['annual_return'].mean():.4f}")
    print(f"Average Annual Volatility: {cluster_data['annual_volatility'].mean():.4f}")
    
    sample_stocks = cluster_data.head(5).index.tolist()
    print(f"Sample stocks: {', '.join(sample_stocks)}")
